# **Prophet Model Development and Evaluation Script**

---

## Modeling Pipeline

- **Baseline Model Development:** A standard Prophet model was trained using default/reasonable hyperparameters to establish a performance benchmark.

- **Hyperparameter Optimization:** Each baseline model was optimized using two complementary techniques:
  - ***`ChangePoint Prior Scale Tuning`***: Prophet-specific optimization that controls trend flexibility by systematically tuning `changepoint_prior_scale` and `changepoint_range` — the two most impactful parameters for capturing structural breaks in gold price returns.
  - ***`Manual GridSearch`***: A comprehensive grid search over 3-5 key Prophet hyperparameters (`changepoint_prior_scale`, `seasonality_prior_scale`, `changepoint_range`, `seasonality_mode`) to find the best parameter combination.

- **Cross-Validation and Robustness Assessment:** Each model variant was evaluated using ***`TimeSeriesSplit`*** to preserve temporal order and prevent look-ahead bias. Metrics were aggregated across folds to assess stability.

- **Overfitting Analysis:** A detailed comparison between cross-validation metrics and test set results was conducted. Additional metrics, including ***`RMSE ratio`*** and ***`R2 gap`***, were computed to quantify overfitting and assess model generalization. ***`Directional accuracy`*** and ***`financial metrics`*** (Sharpe Ratio, Max Drawdown) were also calculated for trading-relevant evaluation.

---

## Optimization Techniques Selection Rationale

From the three candidate techniques (ChangePoint, Seasonality_Prior, Manual GridSearch), the following two were selected as **best for Prophet**:

1. **ChangePoint Prior Scale Tuning** — The `changepoint_prior_scale` is the single most impactful Prophet hyperparameter. It controls how flexible the trend is at changepoints. Too high → overfitting (chasing noise); too low → underfitting (missing real regime shifts). For gold price returns that exhibit structural breaks, tuning this parameter is essential.

2. **Manual GridSearch** — This covers multiple key Prophet parameters systematically, including `seasonality_prior_scale` (which the standalone Seasonality_Prior technique would tune). GridSearch provides a broader, more robust optimization by exploring parameter interactions.

**Why Seasonality_Prior was not selected as a standalone:** It is subsumed by GridSearch, and for daily financial return data, seasonality effects are typically weaker than trend changepoint effects, making it less impactful as a standalone technique.

---

## Persisted Artifacts

To ensure reproducibility, transparency, and extendability, the following artifacts have been saved for **each model**:

- **Optimized Model Performance:** Individual CSV files capturing the performance of each model variant:
    - ***Prophet (Baseline)***
    - ***Prophet (ChangePoint Tuning)***
    - ***Prophet (Manual GridSearch)***

- **Best Variation Performance:** A CSV file containing only the metrics of the best-performing variation per model.

- **Summary of Model Performance:** A consolidated, extendable CSV file (`AllModel_OverallPerformance.csv`) including:
    - Cross-validation results (`CV MSE`, `CV MAE`, `CV RMSE`, `CV R2`, `CV MAPE`)
    - Test set results (`Test MSE`, `Test MAE`, `Test RMSE`, `Test R2`, `Test MAPE`)
    - Financial metrics (`Sharpe Ratio`, `Sortino Ratio`, `Max Drawdown`, `Directional Accuracy`)
    - Overfitting metrics (`R2 gap`, `RMSE ratio`)
    - Overfitting status and model generalization label

- **Overfitting DataFrame:** An extendable CSV (`AllModel_OverfittingAnalysis.csv`) capturing overfitting analysis metrics across all models and variations.

- **Best Model per Algorithm:** The serialized best-performing variant of each algorithm for ensemble consideration or deployment.

- **Model Comparison:** A summary notebook or script that loads `AllModel_OverallPerformance.csv` and generates publication-ready comparison visualizations.

Together, these artifacts provide a complete, reproducible record of the modeling process, facilitating model tracking, comparison, selection, and deployment.

In [1]:
""" Configure the utilities module path for imports """
import sys
import os
from pathlib import Path

# get project root as parent of current working directory
PROJECT_ROOT = Path(os.getcwd()).parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
# artifacts root
DATA_ROOT = PROJECT_ROOT / "data"
FEATURE_ROOT = PROJECT_ROOT / "artifacts" / "FeatureSelection"
FIGURE_ROOT = PROJECT_ROOT / "visualizations" / "ModelEvaluation"
MODEL_ROOT = PROJECT_ROOT / "artifacts" / "Models"
PERFORMANCE_ROOT = PROJECT_ROOT / "artifacts" / "ModelPerformance"
MODEL_CHECKPOINT = MODEL_ROOT / "Checkpoints"

In [3]:
# records and calculations
import pandas as pd
import numpy as np

# visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# Prophet model
from prophet import Prophet
from prophet.serialize import model_to_json, model_from_json
from prophet.diagnostics import cross_validation, performance_metrics as prophet_performance

# metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

import json
import gc

# avoid minor warnings
import warnings
warnings.filterwarnings('ignore')

# import utilities
from src.utilities import Evaluator, DataHandler, ModelPersister

/media/tshihab07/New Volume/03. Python Programming/09 Data Science/02 MachineLearning/ML2026_307_20260303_GoldPricePrediction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# **Load Dataset and Artifact**

In [4]:
df, x, y = DataHandler.load_dataset(DATA_ROOT / "gold_price_engineered.csv", target_col="target")
artifacts = DataHandler.load_artifacts(FEATURE_ROOT, cv_method='tscv')

In [5]:
# check dataset loading
df.head()

,Date,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return,SPX_Return_lag1,SPX_Return_lag2,SPX_Return_lag3,SPX_Return_lag4,...,EURUSD_Return_lag5,GLD_Return_lag1,GLD_Return_lag2,GLD_Return_lag3,GLD_Return_lag4,GLD_Return_lag5,rolling_mean,rolling_std,momentum,target
0,2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339,0.013624,-0.018352,0.003223,-0.024552,...,0.001902,-0.002650,0.023711,-0.004229,-0.005142,0.008367,0.006266,0.014169,0.011275,0.003739
1,2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739,0.007948,0.013624,-0.018352,0.003223,...,0.000679,0.019642,-0.002650,0.023711,-0.004229,-0.005142,0.008043,0.012879,0.008881,0.010838
2,2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337,-0.013595,0.007948,0.013624,-0.018352,...,-0.004875,0.003739,0.019642,-0.002650,0.023711,-0.004229,0.011056,0.010901,0.015066,-0.017311
3,2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499,0.010871,-0.013595,0.007948,0.013624,...,0.060478,0.010838,0.003739,0.019642,-0.002650,0.023711,0.002852,0.013993,-0.041022,-0.014661
4,2008-01-16,-0.005612,-0.014661,-0.012778,-0.011368,-0.009326,-0.024925,0.010871,-0.013595,0.007948,...,-0.058245,-0.017311,0.010838,0.003739,0.019642,-0.002650,0.000449,0.016053,-0.012010,-0.002307


In [6]:
# check input features
x.head()

,Date,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return,SPX_Return_lag1,SPX_Return_lag2,SPX_Return_lag3,SPX_Return_lag4,...,EURUSD_Return_lag4,EURUSD_Return_lag5,GLD_Return_lag1,GLD_Return_lag2,GLD_Return_lag3,GLD_Return_lag4,GLD_Return_lag5,rolling_mean,rolling_std,momentum
0,2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339,0.013624,-0.018352,0.003223,-0.024552,...,0.000679,0.001902,-0.002650,0.023711,-0.004229,-0.005142,0.008367,0.006266,0.014169,0.011275
1,2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739,0.007948,0.013624,-0.018352,0.003223,...,-0.004875,0.000679,0.019642,-0.002650,0.023711,-0.004229,-0.005142,0.008043,0.012879,0.008881
2,2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337,-0.013595,0.007948,0.013624,-0.018352,...,0.060478,-0.004875,0.003739,0.019642,-0.002650,0.023711,-0.004229,0.011056,0.010901,0.015066
3,2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499,0.010871,-0.013595,0.007948,0.013624,...,-0.058245,0.060478,0.010838,0.003739,0.019642,-0.002650,0.023711,0.002852,0.013993,-0.041022
4,2008-01-16,-0.005612,-0.014661,-0.012778,-0.011368,-0.009326,-0.024925,0.010871,-0.013595,0.007948,...,0.009339,-0.058245,-0.017311,0.010838,0.003739,0.019642,-0.002650,0.000449,0.016053,-0.012010


In [7]:
# check target feature
y.head()

0    0.003739
1    0.010838
2   -0.017311
3   -0.014661
4   -0.002307
Name: target, dtype: float64

In [8]:
# load train/test split data
x_train, x_test, y_train, y_test = artifacts['x_train'], artifacts['x_test'], artifacts['y_train'], artifacts['y_test']
cv = artifacts['cv']

# **Configuration Setup**

In [ ]:
# random seed
SEED = 42
np.random.seed(SEED)

In [ ]:
# Prophet baseline hyperparameters
BASELINE_PARAMS = {
    'changepoint_prior_scale': 0.05,
    'seasonality_prior_scale': 10.0,
    'changepoint_range': 0.8,
    'seasonality_mode': 'multiplicative',
    'daily_seasonality': False,
    'weekly_seasonality': True,
    'yearly_seasonality': True,
}

In [ ]:
# regressor columns (all features except 'ds' and 'y')
REGRESSOR_COLS = [col for col in train_prophet.columns if col not in ['ds', 'y']]

In [ ]:
print("Total regressors:", len(REGRESSOR_COLS))

# **Baseline Modeling**

In [ ]:
# Helper Function
def build_prophet_model(params, regressor_cols=None):

    model = Prophet(
        changepoint_prior_scale=params.get('changepoint_prior_scale', 0.05),
        seasonality_prior_scale=params.get('seasonality_prior_scale', 10.0),
        changepoint_range=params.get('changepoint_range', 0.8),
        seasonality_mode=params.get('seasonality_mode', 'multiplicative'),
        daily_seasonality=params.get('daily_seasonality', False),
        weekly_seasonality=params.get('weekly_seasonality', True),
        yearly_seasonality=params.get('yearly_seasonality', True),
    )
    
    # add regressors
    if regressor_cols:
        for reg in regressor_cols:
            model.add_regressor(reg, standardize='auto')
    
    return model


def train_and_predict(model, train_df, test_df):
    model.fit(train_df)
    train_forecast = model.predict(train_df)
    test_forecast = model.predict(test_df)
    return model, train_forecast['yhat'].to_numpy(), test_forecast['yhat'].to_numpy()

In [ ]:
# train baseline model
baseline_model = build_prophet_model(BASELINE_PARAMS, regressor_cols=REGRESSOR_COLS)
baseline_model.fit(train_prophet)

## Apply Model to Make Prediction

In [ ]:
# baseline prediction on train and test set
y_train_pred_baseline = baseline_model.predict(train_prophet)['yhat'].to_numpy()
y_test_pred_baseline = baseline_model.predict(test_prophet)['yhat'].to_numpy()

## Evaluate Model Performance

In [ ]:
# calculate metrics
y_train_true = train_prophet['y'].to_numpy(dtype=float)
y_test_true = test_prophet['y'].to_numpy(dtype=float)

# Prophet predict() returns a forecast DataFrame; metrics need only the numeric yhat values.
if isinstance(y_train_pred_baseline, pd.DataFrame):
    y_train_pred_baseline = y_train_pred_baseline['yhat'].to_numpy(dtype=float)
else:
    y_train_pred_baseline = np.asarray(y_train_pred_baseline, dtype=float)

if isinstance(y_test_pred_baseline, pd.DataFrame):
    y_test_pred_baseline = y_test_pred_baseline['yhat'].to_numpy(dtype=float)
else:
    y_test_pred_baseline = np.asarray(y_test_pred_baseline, dtype=float)

train_metrics_baseline = Evaluator.calculate_metrics(y_train_true, y_train_pred_baseline)
test_metrics_baseline = Evaluator.calculate_metrics(y_test_true, y_test_pred_baseline)

In [ ]:
# Directional accuracy
train_acc_baseline = Evaluator.directional_accuracy(y_train_true, y_train_pred_baseline)
test_acc_baseline = Evaluator.directional_accuracy(y_test_true, y_test_pred_baseline)

In [ ]:
# Financial metrics
fin_baseline = Evaluator.financial_metrics('Prophet (Baseline)', y_test_true, y_test_pred_baseline)
display(fin_baseline)

In [ ]:
# Performance table
baseline_perf = Evaluator.performance_table(
    train_metrics_baseline + [train_acc_baseline],
    test_metrics_baseline + [test_acc_baseline]
)
print("Prophet - Baseline Modeling Performance")
display(baseline_perf)

# **Optimization 1: ChangePoint Prior Scale Tuning**

The `changepoint_prior_scale` is the single most important Prophet hyperparameter. It controls the flexibility of the trend at changepoints:
- **Low values** (0.001-0.01): Rigid trend, may underfit — misses real structural breaks
- **High values** (0.5-1.0): Flexible trend, may overfit — chases noise
- **Moderate values** (0.05-0.1): Balanced — typically a good starting range

We also tune `changepoint_range` which controls how much of the history is used to detect changepoints:
- Default is 0.8 (first 80% of data)
- For financial data, we test 0.6-0.95

This is a sequential, step-by-step manual tuning approach — tune one parameter at a time, lock the best, move to the next.

In [ ]:
def run_prophet_experiment(params, train_df, test_df, experiment_name="experiment", regressor_cols=None):
    gc.collect()
    
    model = build_prophet_model(params, regressor_cols=regressor_cols)
    model, y_train_pred, y_test_pred = train_and_predict(model, train_df, test_df)
    
    y_train_true = train_df['y'].values
    y_test_true = test_df['y'].values
    
    train_metrics = Evaluator.calculate_metrics(y_train_true, y_train_pred)
    test_metrics = Evaluator.calculate_metrics(y_test_true, y_test_pred)
    train_acc = Evaluator.directional_accuracy(y_train_true, y_train_pred)
    test_acc = Evaluator.directional_accuracy(y_test_true, y_test_pred)
    
    result = {
        "experiment": experiment_name,
        "changepoint_prior_scale": params.get('changepoint_prior_scale', 0.05),
        "seasonality_prior_scale": params.get('seasonality_prior_scale', 10.0),
        "changepoint_range": params.get('changepoint_range', 0.8),
        "seasonality_mode": params.get('seasonality_mode', 'multiplicative'),
        "train_MSE": train_metrics[0], "train_MAE": train_metrics[1],
        "train_RMSE": train_metrics[2], "train_R2": train_metrics[3],
        "train_MAPE": train_metrics[4], "train_Dir_Acc": train_acc,
        "test_MSE": test_metrics[0], "test_MAE": test_metrics[1],
        "test_RMSE": test_metrics[2], "test_R2": test_metrics[3],
        "test_MAPE": test_metrics[4], "test_Dir_Acc": test_acc,
    }
    
    gc.collect()
    return result, model

## Step 1: Tune Changepoint Prior Scale

In [ ]:
# Changepoint prior scale values to test
CP_VALUES = [0.001, 0.01, 0.05, 0.1, 0.3, 0.5, 0.8]
cp_results = []

for cp_val in CP_VALUES:
    params = BASELINE_PARAMS.copy()
    params['changepoint_prior_scale'] = cp_val
    
    result, _ = run_prophet_experiment(
        params=params,
        train_df=train_prophet,
        test_df=test_prophet,
        experiment_name=f"cp_{cp_val}",
        regressor_cols=REGRESSOR_COLS
    )
    cp_results.append(result)
    print(f"  cp={cp_val:>6} | test_RMSE={result['test_RMSE']:.6f} | test_R2={result['test_R2']:.4f}")

# Find best
best_cp_result = min(cp_results, key=lambda x: x['test_RMSE'])
BEST_CP = best_cp_result['changepoint_prior_scale']
print(f"\n>>> Best changepoint_prior_scale: {BEST_CP} (test_RMSE={best_cp_result['test_RMSE']:.6f})")

## Step 2: Tune Changepoint Range

In [ ]:
# Changepoint range values to test
CR_VALUES = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95]
cr_results = []

for cr_val in CR_VALUES:
    params = BASELINE_PARAMS.copy()
    params['changepoint_prior_scale'] = BEST_CP
    params['changepoint_range'] = cr_val
    
    result, _ = run_prophet_experiment(
        params=params,
        train_df=train_prophet,
        test_df=test_prophet,
        experiment_name=f"cp_{BEST_CP}_cr_{cr_val}",
        regressor_cols=REGRESSOR_COLS
    )
    cr_results.append(result)
    print(f"  cr={cr_val:>4} | test_RMSE={result['test_RMSE']:.6f} | test_R2={result['test_R2']:.4f}")

# Find best
best_cr_result = min(cr_results, key=lambda x: x['test_RMSE'])
BEST_CR = best_cr_result['changepoint_range']
print(f"\n>>> Best changepoint_range: {BEST_CR} (test_RMSE={best_cr_result['test_RMSE']:.6f})")

## Step 3: Fine-Tune Seasonality Prior Scale

In [ ]:
# Seasonality prior scale values to test
SP_VALUES = [0.01, 0.1, 1.0, 5.0, 10.0, 15.0, 20.0]
sp_results = []

for sp_val in SP_VALUES:
    params = BASELINE_PARAMS.copy()
    params['changepoint_prior_scale'] = BEST_CP
    params['changepoint_range'] = BEST_CR
    params['seasonality_prior_scale'] = sp_val
    
    result, _ = run_prophet_experiment(
        params=params,
        train_df=train_prophet,
        test_df=test_prophet,
        experiment_name=f"cp_{BEST_CP}_cr_{BEST_CR}_sp_{sp_val}",
        regressor_cols=REGRESSOR_COLS
    )
    sp_results.append(result)
    print(f"  sp={sp_val:>5} | test_RMSE={result['test_RMSE']:.6f} | test_R2={result['test_R2']:.4f}")

# Find best
best_sp_result = min(sp_results, key=lambda x: x['test_RMSE'])
BEST_SP = best_sp_result['seasonality_prior_scale']
print(f"\n>>> Best seasonality_prior_scale: {BEST_SP} (test_RMSE={best_sp_result['test_RMSE']:.6f})")

## Step 4: Tune Seasonality Mode

In [ ]:
# Seasonality mode test
SM_VALUES = ['additive', 'multiplicative']
sm_results = []

for sm_val in SM_VALUES:
    params = BASELINE_PARAMS.copy()
    params['changepoint_prior_scale'] = BEST_CP
    params['changepoint_range'] = BEST_CR
    params['seasonality_prior_scale'] = BEST_SP
    params['seasonality_mode'] = sm_val
    
    result, _ = run_prophet_experiment(
        params=params,
        train_df=train_prophet,
        test_df=test_prophet,
        experiment_name=f"cp_cp{BEST_CP}_cr{BEST_CR}_sp{BEST_SP}_sm_{sm_val}",
        regressor_cols=REGRESSOR_COLS
    )
    sm_results.append(result)
    print(f"  sm={sm_val:>14} | test_RMSE={result['test_RMSE']:.6f} | test_R2={result['test_R2']:.4f}")

# Find best
best_sm_result = min(sm_results, key=lambda x: x['test_RMSE'])
BEST_SM = best_sm_result['seasonality_mode']
print(f"\n>>> Best seasonality_mode: {BEST_SM} (test_RMSE={best_sm_result['test_RMSE']:.6f})")

## Build Final ChangePoint-Tuned Model

In [ ]:
# Final ChangePoint-Tuned configuration
CP_CONFIG = {
    'changepoint_prior_scale': BEST_CP,
    'seasonality_prior_scale': BEST_SP,
    'changepoint_range': BEST_CR,
    'seasonality_mode': BEST_SM,
    'daily_seasonality': False,
    'weekly_seasonality': True,
    'yearly_seasonality': True,
}

print("ChangePoint-Tuned Configuration:")
for k, v in CP_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Build and train ChangePoint-Tuned model
cp_model = build_prophet_model(CP_CONFIG, regressor_cols=REGRESSOR_COLS)
cp_model, y_train_pred_cp, y_test_pred_cp = train_and_predict(
    cp_model, train_prophet, test_prophet
)

print("ChangePoint-Tuned Prophet model trained successfully.")

In [ ]:
# Evaluate ChangePoint-Tuned model
train_metrics_cp = Evaluator.calculate_metrics(y_train_true, y_train_pred_cp)
test_metrics_cp = Evaluator.calculate_metrics(y_test_true, y_test_pred_cp)

train_acc_cp = Evaluator.directional_accuracy(y_train_true, y_train_pred_cp)
test_acc_cp = Evaluator.directional_accuracy(y_test_true, y_test_pred_cp)

fin_cp = Evaluator.financial_metrics('Prophet (ChangePoint)', y_test_true, y_test_pred_cp)
display(fin_cp)

cp_perf = Evaluator.performance_table(
    train_metrics_cp + [train_acc_cp],
    test_metrics_cp + [test_acc_cp]
)
print("Prophet - ChangePoint Tuning Performance")
display(cp_perf)

# **Optimization 2: Manual GridSearch**

A comprehensive grid search over 4 key Prophet hyperparameters:
1. `changepoint_prior_scale` — trend flexibility
2. `seasonality_prior_scale` — seasonality strength
3. `changepoint_range` — fraction of history for changepoint detection
4. `seasonality_mode` — additive vs multiplicative

This explores parameter interactions that sequential tuning may miss.

In [ ]:
# GridSearch parameter space
GRID_SPACE = {
    'changepoint_prior_scale': [0.001, 0.01, 0.05, 0.1, 0.3, 0.5],
    'seasonality_prior_scale': [0.01, 0.1, 1.0, 10.0, 15.0],
    'changepoint_range': [0.7, 0.8, 0.9, 0.95],
    'seasonality_mode': ['additive', 'multiplicative'],
}

# Calculate total combinations
total_combo = 1
for v in GRID_SPACE.values():
    total_combo *= len(v)
print(f"Total grid combinations: {total_combo}")

In [ ]:
import itertools

def ManualGridSearch(grid_space, train_df, test_df, regressor_cols=None, seed=SEED):
    """
    Perform manual grid search over Prophet hyperparameters.
    
    Parameters
    ----------
    grid_space : dict - Parameter grid
    train_df : pd.DataFrame - Training data
    test_df : pd.DataFrame - Test data
    regressor_cols : list - Regressor columns
    seed : int - Random seed
    
    Returns
    -------
    list of result dicts
    """
    np.random.seed(seed)
    all_results = []
    
    # Generate all combinations
    keys = list(grid_space.keys())
    values = list(grid_space.values())
    combinations = list(itertools.product(*values))
    
    total = len(combinations)
    print(f"Running {total} grid combinations...")
    
    for i, combo in enumerate(combinations):
        params = dict(zip(keys, combo))
        params['daily_seasonality'] = False
        params['weekly_seasonality'] = True
        params['yearly_seasonality'] = True
        
        exp_name = (f"grid_{i+1}"
                    f"_cp{params['changepoint_prior_scale']}"
                    f"_sp{params['seasonality_prior_scale']}"
                    f"_cr{params['changepoint_range']}"
                    f"_sm{params['seasonality_mode'][:3]}")
        
        try:
            result, _ = run_prophet_experiment(
                params=params,
                train_df=train_df,
                test_df=test_df,
                experiment_name=exp_name,
                regressor_cols=regressor_cols
            )
            result['trial_num'] = i + 1
            all_results.append(result)
        except Exception as e:
            print(f"  Trial {i+1} failed: {e}")
            continue
        
        if (i + 1) % 20 == 0:
            print(f"  Completed {i+1}/{total} trials...")
        
        gc.collect()
    
    print(f"GridSearch complete: {len(all_results)}/{total} successful trials.")
    return all_results

In [ ]:
# Run grid search
grid_results = ManualGridSearch(
    grid_space=GRID_SPACE,
    train_df=train_prophet,
    test_df=test_prophet,
    regressor_cols=REGRESSOR_COLS,
    seed=SEED
)

In [ ]:
# Sort and display results
grid_df = pd.DataFrame(grid_results)
grid_df = grid_df.sort_values(by='test_RMSE', ascending=True).reset_index(drop=True)
print("Top 10 GridSearch Results:")
display(grid_df[['experiment', 'changepoint_prior_scale', 'seasonality_prior_scale',
                   'changepoint_range', 'seasonality_mode', 'test_RMSE', 'test_R2',
                   'test_Dir_Acc']].head(10))

In [ ]:
# Extract best grid config
best_grid = grid_df.iloc[0]
GRID_CONFIG = {
    'changepoint_prior_scale': best_grid['changepoint_prior_scale'],
    'seasonality_prior_scale': best_grid['seasonality_prior_scale'],
    'changepoint_range': best_grid['changepoint_range'],
    'seasonality_mode': best_grid['seasonality_mode'],
    'daily_seasonality': False,
    'weekly_seasonality': True,
    'yearly_seasonality': True,
}

print("Best GridSearch Configuration:")
for k, v in GRID_CONFIG.items():
    print(f"  {k}: {v}")

## Build Final GridSearch Model

In [ ]:
# Build and train GridSearch-optimized model
grid_model = build_prophet_model(GRID_CONFIG, regressor_cols=REGRESSOR_COLS)
grid_model, y_train_pred_grid, y_test_pred_grid = train_and_predict(
    grid_model, train_prophet, test_prophet
)

print("GridSearch-optimized Prophet model trained successfully.")

In [ ]:
# Evaluate GridSearch model
train_metrics_grid = Evaluator.calculate_metrics(y_train_true, y_train_pred_grid)
test_metrics_grid = Evaluator.calculate_metrics(y_test_true, y_test_pred_grid)

train_acc_grid = Evaluator.directional_accuracy(y_train_true, y_train_pred_grid)
test_acc_grid = Evaluator.directional_accuracy(y_test_true, y_test_pred_grid)

fin_grid = Evaluator.financial_metrics('Prophet (GridSearch)', y_test_true, y_test_pred_grid)
display(fin_grid)

grid_perf = Evaluator.performance_table(
    train_metrics_grid + [train_acc_grid],
    test_metrics_grid + [test_acc_grid]
)
print("Prophet - GridSearch Performance")
display(grid_perf)

# **Cross-Validation and Robustness Assessment**

Each model variant is evaluated using `TimeSeriesSplit` to preserve temporal order and prevent look-ahead bias. Metrics are aggregated across folds to assess stability.

In [ ]:
def prophet_cv_evaluate(params, df_full, cv, regressor_cols=None, verbose=0):
    """
    Cross-validate Prophet model using TimeSeriesSplit.
    
    Parameters
    ----------
    params : dict - Prophet hyperparameters
    df_full : pd.DataFrame - Full dataset in Prophet format (ds, y, regressors)
    cv : TimeSeriesSplit - Cross-validation splitter
    regressor_cols : list - Regressor column names
    verbose : int - Verbosity level
    
    Returns
    -------
    dict: Average CV metrics
    """
    metrics_list = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(df_full)):
        gc.collect()
        
        fold_train = df_full.iloc[train_idx].copy()
        fold_val = df_full.iloc[val_idx].copy()
        
        try:
            model = build_prophet_model(params, regressor_cols=regressor_cols)
            model.fit(fold_train)
            
            val_pred = model.predict(fold_val)
            y_val_true = fold_val['y'].values
            y_val_pred = val_pred['yhat'].values
            
            fold_mse = mean_squared_error(y_val_true, y_val_pred)
            fold_mae = mean_absolute_error(y_val_true, y_val_pred)
            fold_rmse = np.sqrt(fold_mse)
            fold_r2 = r2_score(y_val_true, y_val_pred)
            fold_mape = Evaluator.safe_mape(y_val_true, y_val_pred)
            fold_dir = Evaluator.directional_accuracy(y_val_true, y_val_pred)
            
            metrics_list.append({
                "MSE": fold_mse, "MAE": fold_mae, "RMSE": fold_rmse,
                "R2": fold_r2, "MAPE": fold_mape, "Dir_Acc": fold_dir
            })
            
            if verbose:
                print(f"  Fold {fold_idx + 1}: RMSE={fold_rmse:.4f}, R2={fold_r2:.4f}, Dir_Acc={fold_dir:.2f}%")
                
        except Exception as e:
            print(f"  Fold {fold_idx + 1} failed: {e}")
            continue
        
        gc.collect()
    
    if not metrics_list:
        return {
            'CV MSE': np.nan, 'CV MAE': np.nan, 'CV RMSE': np.nan,
            'CV R2': np.nan, 'CV MAPE': np.nan, 'CV Directional Accuracy (%)': np.nan
        }
    
    metrics_df = pd.DataFrame(metrics_list)
    return {
        'CV MSE': metrics_df['MSE'].mean(),
        'CV MAE': metrics_df['MAE'].mean(),
        'CV RMSE': metrics_df['RMSE'].mean(),
        'CV R2': metrics_df['R2'].mean(),
        'CV MAPE': metrics_df['MAPE'].mean(),
        'CV Directional Accuracy (%)': metrics_df['Dir_Acc'].mean(),
    }

In [ ]:
# Prepare full dataset for CV (combine train + test in Prophet format)
full_prophet = pd.concat([train_prophet, test_prophet], axis=0).reset_index(drop=True)
print(f"Full dataset shape for CV: {full_prophet.shape}")

In [ ]:
# CV for Baseline
print("Running CV for Prophet (Baseline)...")
cv_baseline = prophet_cv_evaluate(
    params=BASELINE_PARAMS, df_full=full_prophet, cv=cv,
    regressor_cols=REGRESSOR_COLS, verbose=1
)

In [ ]:
# CV for ChangePoint-Tuned
print("\nRunning CV for Prophet (ChangePoint Tuning)...")
cv_cp = prophet_cv_evaluate(
    params=CP_CONFIG, df_full=full_prophet, cv=cv,
    regressor_cols=REGRESSOR_COLS, verbose=1
)

In [ ]:
# CV for GridSearch
print("\nRunning CV for Prophet (GridSearch)...")
cv_grid = prophet_cv_evaluate(
    params=GRID_CONFIG, df_full=full_prophet, cv=cv,
    regressor_cols=REGRESSOR_COLS, verbose=1
)

In [ ]:
# Build CV results DataFrame
cv_df = pd.DataFrame({
    "Model": ["Prophet (Baseline)", "Prophet (ChangePoint)", "Prophet (GridSearch)"],
    "CV MSE": [cv_baseline['CV MSE'], cv_cp['CV MSE'], cv_grid['CV MSE']],
    "CV MAE": [cv_baseline['CV MAE'], cv_cp['CV MAE'], cv_grid['CV MAE']],
    "CV RMSE": [cv_baseline['CV RMSE'], cv_cp['CV RMSE'], cv_grid['CV RMSE']],
    "CV R2": [cv_baseline['CV R2'], cv_cp['CV R2'], cv_grid['CV R2']],
    "CV MAPE": [cv_baseline['CV MAPE'], cv_cp['CV MAPE'], cv_grid['CV MAPE']],
    "CV Directional Accuracy (%)": [
        cv_baseline['CV Directional Accuracy (%)'],
        cv_cp['CV Directional Accuracy (%)'],
        cv_grid['CV Directional Accuracy (%)']
    ]
}).round(4)

print("Cross-Validation Results:")
display(cv_df)

In [ ]:
# Build performance summary
models = ["Prophet (Baseline)", "Prophet (ChangePoint)", "Prophet (GridSearch)"]
test_metrics = [test_metrics_baseline, test_metrics_cp, test_metrics_grid]
test_dir_acc = [test_acc_baseline, test_acc_cp, test_acc_grid]

performance_summary = Evaluator.summary_builder(models, cv_df, test_metrics, test_dir_acc)

print("Overall Model Performance:")
display(performance_summary)

# **Overfitting Analysis**

In [ ]:
rows = []
idx_count = 0

for _, row in performance_summary.iterrows():
    cv_r2 = row['CV R2']
    test_r2 = row['Test R2']
    cv_rmse = row['CV RMSE']
    test_rmse = row['Test RMSE']
    
    gap, rmse_ratio, overfit_status, gen_status = Evaluator.assess_overfitting(
        cv_r2=cv_r2, test_r2=test_r2, cv_rmse=cv_rmse, test_rmse=test_rmse, tolerance=0.05
    )

    cv_dir_acc = row.get('CV Directional Accuracy (%)', 0)
    test_dir_acc_val = [test_acc_baseline, test_acc_cp, test_acc_grid][idx_count]
    dir_acc_gap = test_dir_acc_val - cv_dir_acc
    
    rows.append({
        "Model": row['Model'],
        "CV RMSE": row['CV RMSE'],
        "CV R2": row['CV R2'],
        "CV Dir Acc (%)": cv_dir_acc,
        "Test RMSE": row['Test RMSE'],
        "Test R2": row['Test R2'],
        "Test Dir Acc (%)": test_dir_acc_val,
        "R2 Gap": gap,
        "RMSE Ratio": rmse_ratio,
        "Dir Acc Gap (%)": dir_acc_gap,
        "Overfitting Status": overfit_status,
        "Model Status (Generalization)": gen_status
    })
    idx_count += 1

overfit_df = pd.DataFrame(rows).round(4)

In [ ]:
print("Prophet - Overfitting Analysis:")
display(overfit_df)

# **Visualization**

In [ ]:
# Standard metrics comparison bar chart
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

model_labels = ["Baseline", "ChangePoint", "GridSearch"]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

metrics = {
    "Test R2": {"title": "Test R2 (higher better)", "direction": "higher", "fmt": ".3f"},
    "Test RMSE": {"title": "Test RMSE (lower better)", "direction": "lower", "fmt": ".4f"},
    "Test MAPE": {"title": "Test MAPE % (lower better)", "direction": "lower", "fmt": ".2f"},
    "Test Dir Acc %": {"title": "Directional Accuracy % (higher better)", "direction": "higher", "fmt": ".1f"}
}

values_dict = {
    "Test R2": performance_summary["Test R2"].values,
    "Test RMSE": performance_summary["Test RMSE"].values,
    "Test MAPE": performance_summary["Test MAPE"].values,
    "Test Dir Acc %": [test_acc_baseline, test_acc_cp, test_acc_grid]
}

for i, (metric, config) in enumerate(metrics.items()):
    ax = axes[i]
    values = values_dict[metric]
    bars = ax.bar(model_labels, values, color=colors, edgecolor='black', linewidth=1.2)
    
    for bar, value in zip(bars, values):
        height = bar.get_height()
        label_pos = height + (max(values) * 0.02) if config["direction"] == "higher" else height + (max(values) * 0.01)
        ax.text(bar.get_x() + bar.get_width()/2., label_pos, f'{value:{config["fmt"]}}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_title(config["title"], fontsize=12, fontweight='bold')
    ax.set_ylabel('Value', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax.set_axisbelow(True)
    
    if metric == "Test R2":
        ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='Moderate threshold')
        ax.legend(fontsize=8, frameon=True)

plt.suptitle("Prophet Model Comparison - Standard Metrics", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
figure_path = FIGURE_ROOT / "05Prophet_Comparison_StandardMetrics.png"
plt.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Financial values extraction
fin_metrics = ['Sharpe Ratio', 'Sortino Ratio', 'Max Drawdown', 'Total Return (%)']

fin_values = {
    "Baseline": [fin_baseline['Sharpe Ratio'].values[0], fin_baseline['Sortino Ratio'].values[0],
                  abs(fin_baseline['Max Drawdown'].values[0]), fin_baseline['Total Return (%)'].values[0]],
    "ChangePoint": [fin_cp['Sharpe Ratio'].values[0], fin_cp['Sortino Ratio'].values[0],
                     abs(fin_cp['Max Drawdown'].values[0]), fin_cp['Total Return (%)'].values[0]],
    "GridSearch": [fin_grid['Sharpe Ratio'].values[0], fin_grid['Sortino Ratio'].values[0],
                    abs(fin_grid['Max Drawdown'].values[0]), fin_grid['Total Return (%)'].values[0]]
}

In [ ]:
def normalize_metrics(values_dict, metrics, higher_is_better=None):
    if higher_is_better is None:
        higher_is_better = [True, True, False, True]
    normalized = {}
    for idx, metric in enumerate(metrics):
        all_values = [values_dict[model][idx] for model in values_dict]
        min_val, max_val = min(all_values), max(all_values)
        range_val = max_val - min_val if max_val != min_val else 1
        for model in values_dict:
            if model not in normalized:
                normalized[model] = []
            val = values_dict[model][idx]
            if higher_is_better[idx]:
                norm_val = (val - min_val) / range_val
            else:
                norm_val = 1 - (val - min_val) / range_val
            normalized[model].append(norm_val)
    return normalized

In [ ]:
# Radar chart for financial metrics
normalized_financial = normalize_metrics(fin_values, fin_metrics)

fig, ax = plt.subplots(1, 1, figsize=(8, 8), subplot_kw=dict(projection='polar'))
angles = np.linspace(0, 2 * np.pi, len(fin_metrics), endpoint=False).tolist()
angles += angles[:1]

for model, color in zip(model_labels, colors):
    values = normalized_financial[model]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=model, color=color, markersize=6)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(fin_metrics, size=11, weight='bold')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
plt.title("Financial Metrics Comparison (Normalized)\nHigher is Better for All",
    fontsize=14, fontweight='bold', y=1.08)
plt.tight_layout()
figure_path = FIGURE_ROOT / "05Prophet_Comparison_FinancialMetrics.png"
plt.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()

# **Save and Persist Artifacts**

In [ ]:
# Merge performance summary with overfitting analysis
final_summary = pd.merge(
    performance_summary,
    overfit_df[['Model', 'R2 Gap', 'RMSE Ratio', 'Dir Acc Gap (%)',
                 'Overfitting Status', 'Model Status (Generalization)']],
    on="Model", how="outer"
)

print("Final Summary:")
display(final_summary)

In [ ]:
# Combine financial metrics
financial_metrics = pd.concat([fin_baseline, fin_cp, fin_grid], axis=0).reset_index(drop=True)
display(financial_metrics)

In [ ]:
# Select best variant
prophet_performance = final_summary.sort_values(
    by=['Test R2', 'Test RMSE'], ascending=[False, True]
)

best_variant = prophet_performance.iloc[0]
best_variant_df = best_variant.to_frame(name='Value').rename_axis('Metrics').reset_index()

print(f"Best variant: {best_variant['Model']}")
display(best_variant_df)

In [ ]:
# Persist all artifacts
persister = ModelPersister(
    model_name="Prophet",
    model_root=MODEL_ROOT,
    performance_root=PERFORMANCE_ROOT
)

# Save overall performance
persister.save_performance(prophet_performance)

# Save best variation
persister.save_performance(best_variant_df, "BestVariation")

# Save financial metrics
persister.save_performance(financial_metrics, "FinancialMetrics")

# Append to aggregated performance
persister.aggregated_performance(prophet_performance, "AllModel_OverallPerformance")

# Append financial metrics to aggregated
persister.aggregated_performance(financial_metrics, "AllModel_FinancialMetrics")

# Append best variant to aggregated
persister.aggregated_performance(prophet_performance.iloc[[0]], "AllModel_BestVariant")

# Append overfitting analysis
persister.append_overfitting(overfit_df)

In [ ]:
# Save best model
best_model_name = prophet_performance.iloc[0]['Model']

if "ChangePoint" in best_model_name:
    best_model = cp_model
elif "GridSearch" in best_model_name:
    best_model = grid_model
else:
    best_model = baseline_model

persister.save_model(best_model)
print(f"\nBest model saved: {best_model_name}")